# METS-R SIM: interactive usage tutorial

This notebook walks through the four main ways a Python client talks to a running METS-R simulator:

1. **Simulation lifecycle** &mdash; configure, launch, reset, reconnect, and terminate a simulation container.
2. **Query APIs** &mdash; read the state of vehicles, roads, signals, and facilities.
3. **Control APIs** &mdash; inject trips, dispatch taxis, adjust bus schedules, teleport vehicles, and drive traffic signals.
4. **Real-time data stream** &mdash; consume the Kafka feed emitted by the simulator for V2X / sensing experiments.

Every section is self-contained: each one (re)starts its own simulator and tears it down at the end, so you can jump straight to the section you care about.

## Setup

This notebook lives under `tutorials/`, but the helper modules (`utils`, `clients`) and the `configs/`, `data/`, and `docker/` folders all sit at the repo root. The cell below switches the working directory to the repo root and makes those packages importable, so the rest of the notebook can keep using short relative paths such as `configs/run_cosim_CARLAT5.json`.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "tutorials":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())


# Section 1: Simulation lifecycle

Every METS-R experiment follows the same skeleton:

1. Read a run configuration JSON.
2. Prepare per-run output directories and launch the Java simulator inside a Docker container.
3. Connect a `METSRClient` over a WebSocket.
4. Advance the clock with `tick()`, interleaving queries and control calls.
5. Optionally `reset()` the world.
6. `terminate()` the simulator when done.

This section uses the CARLA Town05 co-simulation config purely as a small, self-contained map &mdash; the lifecycle commands are the same for every scenario.

## 1.1 Start a simulation

In [ ]:
from utils.util import *
from clients.METSRClient import METSRClient

Load a run config. Setting `verbose=True` makes the client echo every WebSocket message, which is handy while you're learning the API.

In [ ]:
config = read_run_config("configs/run_cosim_CARLAT5.json")

`prepare_sim_dirs` copies the data files referenced in the config into a timestamped output folder and returns the per-instance paths. `run_simulation_in_docker` then launches one container per instance.

In [ ]:
sim_dirs = prepare_sim_dirs(config)
run_simulation_in_docker(config)

print(sim_dirs)

Connect the Python client to the simulator running in the container on port 4000. The constructor blocks until the handshake succeeds.

In [ ]:
client = METSRClient(host="localhost", sim_folder=sim_dirs[0], port=4000)

(Optional) Start the lightweight HTTP server that serves the per-tick trajectory files so you can replay them in the [METS-R Vis](https://engineering.purdue.edu/HSEES/METSRVis/) browser front-end.

In [ ]:
# start the visualization server so the trajectory can be visualized at https://engineering.purdue.edu/HSEES/METSRVis/
client.start_offline_viz()

## 1.2 Advance the clock

`client.tick(N)` asks the server to step `N` ticks (0.1 s each in this config) and blocks until the server acknowledges. You can issue queries or commands between ticks.

In [ ]:
client.tick(1000)

### Generating trips

Passing `-1` as origin / destination asks the simulator to sample them uniformly. This loop injects 50 private-vehicle trips, spaced 10 ticks apart, while advancing the clock.

In [ ]:
# Generating 50 trips in 500 ticks.
for i in range(500):
      if(i % 10 == 0):
            client.generate_trip(i, -1, -1)
      client.tick(1)

## 1.3 Reset the world

`reset()` rolls the simulator back to `tick 0` **without** relaunching the container. All vehicles, demand, and facility state are cleared.

In [ ]:
# reset the simulation using CARLA Town05's map
client.reset()

Same 50 trips as before, but issued in one batch after each 10-tick window &mdash; significantly fewer round-trips.

In [ ]:
# This is doing the same thing as above, but in a more efficient way
for i in range(50):
      client.generate_trip(i, -1, -1)
      client.tick(10)

## 1.4 Trips between specific roads

`query_road()` with no argument returns every road ID known to the simulator. `generate_trip_between_roads` then lets you dictate the O-D pair rather than sampling it.

In [ ]:
# generate trips between roads
import random
road_ids = client.query_road()
road_ids

In [ ]:
for i in range(500):
      client.tick(10)
      origin_road = random.choice(road_ids['id_list'])
      destination_road = random.choice(road_ids['id_list'])
      client.generate_trip_between_roads(i, origin_road, destination_road)

## 1.5 Clean up

In [ ]:
# stop the visualization server, note visualization server is already stopped when the previous client is closed so nothing is going to be printed
client.stop_viz()

In [ ]:
# terminate the simulation, note if the visualization server is not stopped, it will be automatically stopped when the simulation is terminated, so most of time you don't need to stop the visualization server like above
client.terminate()

# Section 2: Query APIs

Query calls are read-only: they pull state out of the simulator without changing it. We switch to the NYC interactive scenario because it has a full bus schedule, e-taxi fleet, and realistic ride-hail demand &mdash; plenty for the queries to return meaningful data.

## 2.1 Launch the NYC scenario

In [ ]:
from utils.util import *
from clients.METSRClient import METSRClient

In [ ]:
config = read_run_config("configs/run_interactive_NYC.json")

In [ ]:
sim_dirs = prepare_sim_dirs(config)
run_simulation_in_docker(config)

print(sim_dirs)

When `metsr_port` is omitted from the config, `prepare_sim_dirs` picks free ports automatically and stores them on the config object.

In [ ]:
# If no metsr_port is specified, the runner will find the port automatically 
config.ports

Bigger scenarios need a larger request timeout. 300 s is generous for NYC; raise it further if you see timeout errors.

In [ ]:
# Better to  set up a larger timeout (default is 30s) if your map/vehicle number is huge
sim_client = METSRClient(host="localhost", sim_folder=sim_dirs[0], port = config.ports[0], timeout=300) 

Warm the simulation up for 10 minutes of simulated time (3 000 ticks at 0.2 s / tick). `wait_forever=True` prevents timeouts on very long ticks.

In [ ]:
sim_client.tick(3000, wait_forever=True)

The above approach may be less transparent when running a large number of ticks. A more reactive way to run the simulation is as follows.|

In [ ]:
from tqdm import tqdm

for i in tqdm(range(10)):
      sim_client.tick(300, wait_forever=True)

## 2.2 Vehicle queries

`query_bus` / `query_taxi` with no argument return the list of IDs in that fleet. Pass an ID to get the full state object for that vehicle.

In [ ]:
# if no vehicle id is specified, a list of all buses will be returned
sim_client.query_bus()

Single bus &mdash; note the `current_stop` field telling us this vehicle has already reached its second stop.

In [ ]:
# this vehicle has reached the second stop
sim_client.query_bus(4032)

In [ ]:
# similarly, one can query taxis
sim_client.query_taxi()

`state == 0` means the taxi is parked and waiting for a request.

In [ ]:
# This vehicle is in state 0, meaning it is parking
sim_client.query_taxi(0)

In [ ]:
# the definition of the state can be found using
?sim_client.query_taxi

A quick scan for a taxi that is actively carrying passengers.

In [ ]:
# Find a taxi, which is carrying passengers
for i in sim_client.query_taxi()['id_list']:
      res = sim_client.query_taxi(i)
      if res['DATA'][0]['pass_num']>0:
            print(i)
            print(res)
            break

## 2.3 Road network queries

`query_road()` returns all road IDs. Passing a single string returns one road; passing a list returns many at once, which is much cheaper than issuing N round-trips.

In [ ]:
# One can also query the facilities in the simulation
sim_client.query_road()

In [ ]:
# This id can be found in the road csv (or SUMO XML) in the data folder
sim_client.query_road('-1103160164')

In [ ]:
# Also, a list of ID can be used to query multiple roads
sim_client.query_road(sim_client.query_road()['id_list'][:5])

## 2.4 Traffic signal queries

Signals can be queried either by their own ID or by the pair of incoming/outgoing road IDs they govern.

In [ ]:
sim_client.query_signal()

In [ ]:
sim_client.query_signal(2)

In [ ]:
?sim_client.query_signal

In [ ]:
sim_client.query_signal_between_roads('-1001065257', '119628652')

## 2.5 Facility queries

In [ ]:
sim_client.query_chargingStation()

In [ ]:
sim_client.query_chargingStation(-3)

In [ ]:
sim_client.query_zone(131)

## 2.6 Putting it together: trajectory + zone monitoring

We record the position of vehicle `0` every 1 s of simulated time, and sample demand/supply for zone 141 every 60 s. This produces a trajectory we can map and three demand time series.

In [ ]:
xs, ys = [], []
taxi_demand, bus_demand, taxi_stock = [], [], []

from tqdm import tqdm

# Run a 1 hr simulation, record the trajectory every 1s, and monitor the demand every minute
for i in tqdm(range(1800)):
      query_res = sim_client.query_taxi(0)
      xs.append(query_res['DATA'][0]['x'])
      ys.append(query_res['DATA'][0]['y'])

      if i % 60 == 0:
            taxi_demand.append(sim_client.query_zone(141)['DATA'][0]['taxi_demand'])
            bus_demand.append(sim_client.query_zone(141)['DATA'][0]['bus_demand'])
            taxi_stock.append(sim_client.query_zone(141)['DATA'][0]['veh_stock'])

      sim_client.tick(5)

The plotting below needs `contextily` and `geopandas`. Uncomment the `pip install` lines on first run.

In [ ]:
# !pip install contextily
# !pip install geopandas

In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx
import geopandas as gpd

# Example 1: Collect and visualize the trajectory of one vehicle
fig, ax = plt.subplots()

# add the NYC map
gdf = gpd.GeoDataFrame(geometry=gpd.points_from_xy(xs, ys), crs='EPSG:4326').to_crs(epsg=3857)
gdf.plot(ax = ax)
ax.set_title("Trajectory of Vehicle 0")
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

ax.set_axis_off()

plt.show()

# Example 2: Minitor the state of a zone
fig, axs = plt.subplots(1,3, figsize=(15,3), sharex = True)

# The first two hours are 0-2:00, so the demand is low
axs[0].plot(taxi_demand)
axs[0].set_title("Taxi Demand")

axs[1].plot(bus_demand)
axs[1].set_title("Bus Demand")

axs[2].plot(taxi_stock)
axs[2].set_title("Taxi Stock")
for i in range(3):
      axs[i].set_xticks(range(0, 31, 5))
      axs[i].set_xticklabels(range(0, 31, 5))
      axs[i].set_xlabel("Time (min)")

plt.show()

Tear the simulator down before moving on to the next section.

In [ ]:
sim_client.terminate()

## 2.7 Real-time visualization

Enabled by the Query API, one can start the websocket connection so you can visualize real-time simulation states in the [METS-R Vis](https://engineering.purdue.edu/HSEES/METSRVis/).

In [ ]:
from utils.util import *
from clients.METSRClient import METSRClient

config = read_run_config("configs/run_interactive_NYC.json")

sim_dirs = prepare_sim_dirs(config)
run_simulation_in_docker(config)

sim_client = METSRClient(host="localhost", sim_folder=sim_dirs[0], port = config.ports[0], timeout=300) 

In [ ]:
sim_client.start_viz()

In [ ]:
from tqdm import tqdm

In [ ]:
for i in range(100):
      sim_client.generate_trip(i, -1, -1)

In  [METS-R Vis](https://engineering.purdue.edu/HSEES/METSRVis/), click the Stream button.

In [ ]:
info = sim_client.render()
info

In [ ]:
for _ in tqdm(range(300)):
      sim_client.tick(10, wait_forever=True)
      info = sim_client.render()

In [ ]:
sim_client.terminate()

# Section 3: Control APIs

Control calls let you inject, reroute, or override entities in the simulation. They always produce a response that tells you whether the command was accepted. We reuse the NYC scenario from Section 2.

## 3.1 Launch

In [ ]:
from utils.util import *
from clients.METSRClient import METSRClient

In [ ]:
config = read_run_config("configs/run_interactive_NYC.json")

In [ ]:
sim_dirs = prepare_sim_dirs(config)
run_simulation_in_docker(config)

print(sim_dirs)

In [ ]:
client = METSRClient(host="localhost", sim_folder=sim_dirs[0], port=config.ports[0], timeout=300)

Start with an empty private-vehicle fleet &mdash; we'll populate it via control calls below.

In [ ]:
# no private vehicle is in the simulation yet
client.query_vehicle(private_veh = True)['private_vids']

## 3.2 Private vehicle trips

In [ ]:
# generate a private vehicle trip
client.generate_trip(0, -1, -1)

State `8` on a private vehicle means it is currently serving a trip.

In [ ]:
# state 8 means the vehicle is performing a private trip
client.query_vehicle(0, private_veh = True)

We can also dictate the O-D pair for the private trip.

In [ ]:
# can also generate a private vehicle trip between two roads
import random
road_ids = client.query_road()
origin_road = random.choice(road_ids['id_list'])
destination_road = random.choice(road_ids['id_list'])

client.generate_trip_between_roads(1, origin_road, destination_road)

In [ ]:
client.query_vehicle(1, private_veh = True)

## 3.3 Taxi control

In [ ]:
client.query_taxi(100)

`dispatch_taxi(vehID, reqID)` forces taxi 100 to pick up a passenger with ID reqID.

In [ ]:
client.tick(300)

In [ ]:
client.query_pending_requests()

In [ ]:
# generate a taxi trip with specific vehicles
client.dispatch_taxi(100, 12)

The taxi is now in state `6` (on a pickup leg); its reported destination is still the pickup zone.

In [ ]:
# We see the veh's destination is still 140, this is because it is in a pickup trip (state 6)
client.query_taxi(100)

In [ ]:
# let's tick the simulation for 360s 
client.tick(1800)

After 6 minutes of simulated time, the taxi has completed the pickup and is now driving the passenger toward the drop-off.

In [ ]:
# aha
client.query_taxi(100)

### Taxi requests without a specific vehicle

`add_taxi_requests(origZone, destZone, numPassengers)` feeds a new request into the dispatcher, which will assign it to some available taxi at the next planning cycle (every 60 s in this scenario).

In [ ]:
client.query_zone(140)

In [ ]:
# generate a taxi request without specific vehicles
client.add_taxi_requests(140, 181, 3, 3000)

In [ ]:
# this will take effect when the next round of demand processing, which happens every 60s in this scenario
client.tick(600)

In [ ]:
client.query_zone(140)

> Tip: `add_taxi_requests` also accepts road-level origin/destination IDs if you want pickup/drop-off to be deterministic rather than sampled inside the zone.

## 3.4 Bus schedules

Bus control follows the same pattern: first discover the vehicle, then send it an assignment.

In [ ]:
buses = client.query_bus()['id_list']
buses

Tick forward until at least one bus has started its route (`route != -1`).

In [ ]:
# Now, let's tick a little bit longer to let the bus operate
flag = True
while flag:
      client.tick(100)
      for bus in buses:
            if client.query_bus(bus)['DATA'][0]['route'] != -1:
                  print("Tick: ", client.current_tick)
                  print(bus)
                  flag = False
                  break
      if client.current_tick>=20000:
            break

In [ ]:
# Our target is this bus
client.query_bus(4032)

In [ ]:
client.query_bus_route('1640002')

In [ ]:
client.add_bus_requests(157, 228, '1640002', 10, 3000)

In [ ]:
client.insert_bus_stop(4032, "1640002", 31, '1189567313#8', 3)

In [ ]:
client.query_bus(4032)

In [ ]:
client.tick(1500)

In [ ]:
# request already picked up
client.query_bus(4032)

## 3.5 Teleportation & acceleration override

These are primarily useful for scripted trace replays and adversarial scenarios. Teleportation must stay within the same road; the fourth parameter is the distance (m) from the target lane's downstream end.

In [ ]:
# let's teleport the bus!
client.query_vehicle(4032, private_veh = False)

In [ ]:
# teleport a vehicle to a specific location (must within the same road), the fourth parameter is the target distance to the next intersection
client.teleport_trace_replay_vehicle(4032, roadID='46522003#0', laneID=0, dist=5.0, private_veh = False)

In [ ]:
# let's check the teleported vehicle
client.query_vehicle(4032, private_veh = False)

`control_vehicle(vehID, accel_m_s2)` forces the longitudinal acceleration for the next planning cycle.

In [ ]:
# force the vehicle to take the specified acceleration
client.control_vehicle(4032, 2.0, private_veh = False)

## 3.6 Traffic signal control

The signal API exposes three levels of granularity: single-phase changes, phase-duration updates, and full phase plans (with sub-tick precision when you need it). The cell below demonstrates all of them on one intersection.

In [ ]:
# First, let's query the signal to get its current state
signal_info = client.query_signal_between_roads('-1001065257', '119628652')
print("Current signal state:", signal_info)

signal_id = signal_info['DATA'][0]['signalID']
print(f"\nSignal ID: {signal_id}")
print(f"Current tick: {client.current_tick}\n")

In [ ]:
# Example 1: Update signal phase (0=Green, 1=Yellow, 2=Red)
# Change the signal to green phase
result = client.update_signal(signal_id, targetPhase=0)
print("\n1. Updated signal to Green phase:")
print(result)

# Query again to see the updated state
updated_info = client.query_signal(signal_id)
print("\nSignal state after phase update:", updated_info)

In [ ]:
# Example 2: Update signal timing (green, yellow, red durations in seconds)
# Set green=30s, yellow=5s, red=25s
result = client.update_signal_timing(signal_id, greenTime=30, yellowTime=5, redTime=25)
print("\n3. Updated signal timing (Green:30s, Yellow:5s, Red:25s):")
print(result)

In [ ]:
# Example 3: Set a complete phase plan (timing + starting state + offset)
# This sets the entire phase plan: green=40s, yellow=6s, red=30s, start at green, with 5s offset
result = client.set_signal_phase_plan(signal_id, greenTime=40, yellowTime=6, redTime=30, 
                                        startPhase=0, phaseOffset=5)
print("\n4. Set complete phase plan (Green:40s, Yellow:6s, Red:30s, Start:Green, Offset:5s):")
print(result)

In [ ]:
# Example 5: Set phase plan with tick-level precision (more precise control)
# Assuming 0.1s per tick: green=400 ticks (40s), yellow=60 ticks (6s), red=300 ticks (30s)
result = client.set_signal_phase_plan_ticks(signal_id, greenTicks=400, yellowTicks=60, 
                                                redTicks=300, startPhase=0, tickOffset=50)
print("\n5. Set phase plan with tick precision (Green:400 ticks, Yellow:60 ticks, Red:300 ticks):")
print(result)

In [ ]:
client.terminate()

# Section 4: Real-time data stream

METS-R can publish a Kafka feed of per-tick Basic Safety Messages (BSMs) or travel time/energy consumption for a configurable subset of vehicles.

Running Kafka requires the auxiliary docker-compose stack under `docker/`. We switch back to the small CARLA Town05 map so this section is quick to run.

## 4.1 Launch Kafka and the simulator

In [ ]:
from utils.util import *
from clients.METSRClient import METSRClient
from clients.KafkaDataProcessor import KafkaDataProcessor

In [ ]:
config = read_run_config("configs/run_cosim_CARLAT5.json")

Bring Kafka + Zookeeper up, give them 10 s to finish starting, then prepare the simulation output folder and launch the simulator.

In [ ]:
os.chdir("docker")
os.system("docker-compose up -d")
os.chdir("..")

time.sleep(10) # wait 10s for the Kafka servers to be up

sim_dirs = prepare_sim_dirs(config)
run_simulation_in_docker(config)

print(sim_dirs)

## 4.2 Connect client and data processor

In [ ]:
sim_client = METSRClient(host="localhost", sim_folder=sim_dirs[0], port=4000)

kafkaDataProcessor = KafkaDataProcessor(config)

## 4.3 Generate traffic and subscribe to V2X

Create 100 private-vehicle trips, then promote 10 of them to V2X vehicles so their BSMs start flowing into Kafka.

In [ ]:
# First, generate 100 trips
sim_client.generate_trip(list(range(100)), -1, -1)


In [ ]:
# Set up 10 vehicles as V2X vehicles
sim_client.update_vehicle_sensor_type(list(range(10, 20)), 1, True)

Every iteration advances the sim by 1 s and drains whatever messages arrived during that window.

In [ ]:
# consume the information from the data stream every 1s
for i in range(100):
      sim_client.tick(10)
      res = kafkaDataProcessor.process()
      if res is not None:
            break
      print(res)

Inspect a single BSM.

In [ ]:
res

## 4.4 Tear down

In [ ]:
sim_client.terminate()

In [ ]:
os.chdir("docker")
os.system("docker-compose down")
os.chdir("..")